In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import sys
import os
import numpy as np
pd.set_option('display.max_columns', 1000);

Dashboard Objectives

    Provide interactive visualizations for key patient health indicators.
    Assist nursing students in understanding clinical assessments from OASIS and county level data (Census & SDOH).


In [2]:
import plotly.express as px
import plotly.graph_objects as go
import holoviews as hv
from bokeh.palettes import Category10
import scipy.cluster.hierarchy as sch

from bokeh.plotting import figure

In [3]:
import panel as pn
pn.extension()


In [4]:
from urllib.request import urlopen
import json
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)

counties["features"][0]

{'type': 'Feature',
 'properties': {'GEO_ID': '0500000US01001',
  'STATE': '01',
  'COUNTY': '001',
  'NAME': 'Autauga',
  'LSAD': 'County',
  'CENSUSAREA': 594.436},
 'geometry': {'type': 'Polygon',
  'coordinates': [[[-86.496774, 32.344437],
    [-86.717897, 32.402814],
    [-86.814912, 32.340803],
    [-86.890581, 32.502974],
    [-86.917595, 32.664169],
    [-86.71339, 32.661732],
    [-86.714219, 32.705694],
    [-86.413116, 32.707386],
    [-86.411172, 32.409937],
    [-86.496774, 32.344437]]]},
 'id': '01001'}

In [5]:
df = pd.read_csv('data/dashboardData_ICDSections_v2.csv', low_memory=False)

In [6]:
df

,Beneficiary_ID,Assessment_Effective_Date,Submitted_HIPPS_Code,Facility_Internal_ID,Age,Gender,American_Indian_or_Alaska_Native,Asian,Black_or_African_American,Hispanic_or_Latino,Native_Hawiian_or_Pacific_Islander,Unknown_Race_Ethnicity,White,ByDiscipline,DaysBetweenVisits,Days_Cared_For,READMISSION,Agency_Medicare_Number,Primary_Diagnosis_ICD_10_C_M_Code,Other_Diagnosis_Code_1_ICD_10_C_M,Other_Diagnosis_Code_2_ICD_10_C_M,Other_Diagnosis_Code_3_ICD_10_C_M,Other_Diagnosis_Code_4_ICD_10_C_M,Other_Diagnosis_Code_5_ICD_10_C_M,Weight_in_pounds,Height_in_inches,is_diabetes,is_heart_failure,is_hypertension,is_renal_failure,BMI,BMI_Category,COUNTY_NAME,POP_COU,HOU_COU,POP_URB,POPPCT_URB,POP_RUR,POPPCT_RUR,COUNTYFIPS,ACS_PCT_BACHELOR_DGR,ACS_PCT_COLLEGE_ASSOCIATE_DGR,ACS_PCT_GRADUATE_DGR,ACS_PCT_HS_GRADUATE,ACS_PCT_LT_HS,ACS_PCT_NO_WORK_NO_SCHL_16_19,ACS_PCT_POSTHS_ED,ACS_PCT_VET_BACHELOR,ACS_PCT_VET_COLLEGE,ACS_PCT_VET_HS,ACS_PCT_HH_LIMIT_ENGLISH,ACS_PCT_HH_BROADBAND,ACS_PCT_HH_BROADBAND_ANY,ACS_PCT_HH_BROADBAND_ONLY,ACS_PCT_HH_CELLULAR,ACS_PCT_HH_CELLULAR_ONLY,ACS_PCT_HH_DIAL_INTERNET_ONLY,ACS_PCT_HH_INTERNET,ACS_PCT_HH_INTERNET_NO_SUBS,ACS_PCT_HH_NO_COMP_DEV,ACS_PCT_HH_NO_INTERNET,ACS_PCT_HH_OTHER_COMP,ACS_PCT_HH_OTHER_COMP_ONLY,ACS_PCT_HH_PC,ACS_PCT_HH_PC_ONLY,ACS_PCT_HH_SAT_INTERNET,ACS_PCT_HH_SMARTPHONE,ACS_PCT_HH_SMARTPHONE_ONLY,ACS_PCT_HH_TABLET,ACS_PCT_HH_TABLET_ONLY,ACS_PCT_CHILDREN_GRANDPARENT,ACS_PCT_CHILD_1FAM,ACS_PCT_GRANDP_NO_RESPS,ACS_PCT_GRANDP_RESPS_NO_P,ACS_PCT_GRANDP_RESPS_P,ACS_PCT_HH_1PERS,ACS_PCT_HH_ABOVE65,ACS_PCT_HH_ALONE_ABOVE65,ACS_PCT_HH_KID_1PRNT,ACS_TOT_GRANDCHILDREN_GP,ACS_PCT_HEALTH_INC_138_199,ACS_PCT_HEALTH_INC_200_399,ACS_PCT_HEALTH_INC_ABOVE400,ACS_PCT_HEALTH_INC_BELOW137,ACS_PCT_HH_1FAM_FOOD_STMP,ACS_PCT_HH_FOOD_STMP,ACS_PCT_HH_FOOD_STMP_BLW_POV,ACS_PCT_HH_NO_FD_STMP_BLW_POV,ACS_PCT_HH_PUB_ASSIST,ACS_PCT_INC50,ACS_PCT_INC50_ABOVE65,ACS_PCT_NONVET_POV_18_64,ACS_PCT_PERSON_INC_100_124,ACS_PCT_PERSON_INC_125_199,ACS_PCT_PERSON_INC_ABOVE200,ACS_PCT_PERSON_INC_BELOW99,ACS_PCT_POV_AIAN,ACS_PCT_POV_ASIAN,ACS_PCT_POV_BLACK,ACS_PCT_POV_HISPANIC,ACS_PCT_POV_MULTI,ACS_PCT_POV_NHPI,ACS_PCT_POV_OTHER,ACS_PCT_POV_WHITE,ACS_PCT_VET_POV_18_64,ACS_TOT_CIVIL_NONINST_POP_POV,ACS_TOT_CIVIL_POP_POV,ACS_TOT_POP_POV,ICD_Section,ICD_Range
0,GGGGGGG99DXUNNS,2017-11-29,4BGLS,20709,77,Female,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PT,NaN,56,0,679120,I69.354,I69.311,F03.90,I10.,H35.3290,Z91.81,NaN,NaN,0,0,0,0,0,Underweight,Travis,1290188.0,562488.0,1224251.0,94.89,65937.0,5.11,48453,32.074345,22.720897,18.759759,15.975069,10.125310,1.126621,73.554724,43.738517,32.210448,21.982241,5.364069,77.367655,88.875172,6.101690,82.169241,9.295000,0.120241,88.995310,3.084621,3.860310,6.885931,2.473000,0.005034,85.658207,2.725345,5.117103,90.312759,5.603414,68.051034,0.605414,5.587552,26.428103,31.441345,8.488448,20.070172,28.869621,18.399828,6.166552,14.115345,59.224138,9.056379,25.322103,47.800931,17.476414,15.557483,6.956862,2.961000,8.280103,7.701241,6.646034,4.280931,11.512966,3.100828,10.822793,73.126034,12.605586,5.032621,10.705379,19.303103,14.320379,11.069241,2.068966,14.276517,9.661000,9.426172,4223.255172,3299.406897,4225.820690,"{'Diseases of the eye and adnexa', 'Diseases o...","{'F01-F99', 'Z00-Z99', 'I00-I99', 'H00-H59'}"
1,GGGGGGG99GWNzUW,2017-02-08,1AHKS,696175,74,Female,0.0,0.0,1.0,0.0,0.0,NaN,0.0,RN,NaN,314,0,747199,I25.10,I25.83,I50.42,M12.89,M54.30,R26.89,146.0,65.0,0,0,0,0,25,Overweight,Harris,4731145.0,1842683.0,4677507.0,98.87,53638.0,1.13,48201,20.002284,25.887536,11.996951,22.838174,19.185522,1.939011,57.886619,31.882239,36.757860,28.122410,12.653121,68.342320,85.252104,6.312617,78.048615,14.276799,0.141951,85.394164,2.052950,7.455809,12.283121,2.412104,0.018714,74.575108,3.227824,7.685683,86.899218,12.036313,59.834092,0.734829,8.129937,33.536888,38.805908,10.236151,24.519155,26.830728,21.947275,7.247464,21.881277,93.839029,12.250612,28.372572,34.331007,24.776376,24.694119,13.773768,6.257833,9.230647,14.404757,7.291655,4.5062

In [7]:
df['COUNTYFIPS'] = df['COUNTYFIPS'].astype(int)

In [8]:
df.columns.tolist()

['Beneficiary_ID',
 'Assessment_Effective_Date',
 'Submitted_HIPPS_Code',
 'Facility_Internal_ID',
 'Age',
 'Gender',
 'American_Indian_or_Alaska_Native',
 'Asian',
 'Black_or_African_American',
 'Hispanic_or_Latino',
 'Native_Hawiian_or_Pacific_Islander',
 'Unknown_Race_Ethnicity',
 'White',
 'ByDiscipline',
 'DaysBetweenVisits',
 'Days_Cared_For',
 'READMISSION',
 'Agency_Medicare_Number',
 'Primary_Diagnosis_ICD_10_C_M_Code',
 'Other_Diagnosis_Code_1_ICD_10_C_M',
 'Other_Diagnosis_Code_2_ICD_10_C_M',
 'Other_Diagnosis_Code_3_ICD_10_C_M',
 'Other_Diagnosis_Code_4_ICD_10_C_M',
 'Other_Diagnosis_Code_5_ICD_10_C_M',
 'Weight_in_pounds',
 'Height_in_inches',
 'is_diabetes',
 'is_heart_failure',
 'is_hypertension',
 'is_renal_failure',
 'BMI',
 'BMI_Category',
 'COUNTY_NAME',
 'POP_COU',
 'HOU_COU',
 'POP_URB',
 'POPPCT_URB',
 'POP_RUR',
 'POPPCT_RUR',
 'COUNTYFIPS',
 'ACS_PCT_BACHELOR_DGR',
 'ACS_PCT_COLLEGE_ASSOCIATE_DGR',
 'ACS_PCT_GRADUATE_DGR',
 'ACS_PCT_HS_GRADUATE',
 'ACS_PCT_LT_HS

In [10]:
from shapely.geometry import shape
import geopandas as gpd

In [11]:
features = counties["features"]
geometry = [shape(feature['geometry']) for feature in features]
names = [feature['properties']['NAME'] for feature in features]
fips = [feature['id'] for feature in features]
states = [feature['properties']['STATE'] for feature in features]

geo_df = gpd.GeoDataFrame({
    'County': names,
    'Fips': fips,
    'geometry': geometry,
    'State': states
})
geo_df

,County,Fips,geometry,State
0,Autauga,01001,"POLYGON ((-86.49677 32.34444, -86.7179 32.4028...",01
1,Blount,01009,"POLYGON ((-86.5778 33.76532, -86.75914 33.8406...",01
2,Chambers,01017,"POLYGON ((-85.18413 32.87052, -85.12342 32.772...",01
3,Chilton,01021,"POLYGON ((-86.51734 33.02057, -86.51596 32.929...",01
4,Colbert,01033,"POLYGON ((-88.13999 34.5817, -88.13925 34.5878...",01
...,...,...,...,...
3216,Accomack,51001,"MULTIPOLYGON (((-75.24227 38.02721, -75.29687 ...",51
3217,Bland,51021,"POLYGON ((-81.2251 37.23487, -81.20477 37.2430...",51
3218,Buchanan,51027,"POLYGON ((-81.9683 37.5378, -81.92787 37.51212...",51
3219,Charlotte,51037,"POLYGON ((-78.44332 37.0794, -78.49303 36.8912...",51


In [12]:
geo_df.set_crs('EPSG:4326', allow_override=True, inplace=True)
#source: https://public-rma.fpac.usda.gov/ReportShare//Extranet/SOB/Listings/StateAbbr2014.pdf
data = {
    "State_Abbreviation": [
        "AK", "AL", "AR", "AS", "AS", "AZ", "CA", "CO", "CT", "DC", "DE", "FL", "FM", "GA", "GU", "HI", "IA", "ID", "IL", "IN", "KS", "KY", "LA", 
        "MA", "MD", "ME", "MH", "MI", "MN", "MO", "MP", "MS", "MT", "NC", "ND", "NE", "NH", "NJ", "NM", "NV", "NY", "OH", "OK", "OR", "PA", "PR", 
        "PR", "PW", "RI", "SC", "SD", "TN", "TX", "UM", "UT", "VA", "VI", "VI", "VT", "WA", "WI", "WV", "WY", "ZZ"
    ],
    "State_name": [
        "ALASKA", "ALABAMA", "ARKANSAS", "AMERICAN SAMOA", "ALL STATES", "ARIZONA", "CALIFORNIA", "COLORADO", "CONNECTICUT", "D. OF COLUMBIA",
        "DELAWARE", "FLORIDA", "FS OF MICRONESI", "GEORGIA", "GUAM", "HAWAII", "IOWA", "IDAHO", "ILLINOIS", "INDIANA", "KANSAS", "KENTUCKY",
        "LOUISIANA", "MASSACHUSETTS", "MARYLAND", "MAINE", "MARSHALL ISLAND", "MICHIGAN", "MINNESOTA", "MISSOURI", "NORTHERN MARIAN", 
        "MISSISSIPPI", "MONTANA", "NORTH CAROLINA", "NORTH DAKOTA", "NEBRASKA", "NEW HAMPSHIRE", "NEW JERSEY", "NEW MEXICO", "NEVADA", 
        "NEW YORK", "OHIO", "OKLAHOMA", "OREGON", "PENNSYLVANIA", "PUERTO RICO", "PUERTO RICO", "PALAU", "RHODE ISLAND", "SOUTH CAROLINA", 
        "SOUTH DAKOTA", "TENNESSEE", "TEXAS", "U.S. MINOR OUTL", "UTAH", "VIRGINIA", "VIRGIN ISLANDS", "VIRGIN ISLANDS", "VERMONT", "WASHINGTON", 
        "WISCONSIN", "WEST VIRGINIA", "WYOMING", "FOREIGN"
    ],
    "Code": [
        "02", "01", "05", "60", "98", "04", "06", "08", "09", "11", "10", "12", "64", "13", "66", "15", "19", "16", "17", "18", "20", "21", "22", 
        "25", "24", "23", "68", "26", "27", "29", "69", "28", "30", "37", "38", "31", "33", "34", "35", "32", "36", "39", "40", "41", "42", "72", 
        "43", "70", "44", "45", "46", "47", "48", "74", "49", "51", "78", "52", "50", "53", "55", "54", "56", "99"
    ]
}

codes = pd.DataFrame(data)
codes

,State_Abbreviation,State_name,Code
0,AK,ALASKA,02
1,AL,ALABAMA,01
2,AR,ARKANSAS,05
3,AS,AMERICAN SAMOA,60
4,AS,ALL STATES,98
...,...,...,...
59,WA,WASHINGTON,53
60,WI,WISCONSIN,55
61,WV,WEST VIRGINIA,54
62,WY,WYOMING,56


In [13]:
merged_data = pd.merge(geo_df, codes, right_on=['Code'], left_on=['State'])
merged_data

,County,Fips,geometry,State,State_Abbreviation,State_name,Code
0,Autauga,01001,"POLYGON ((-86.49677 32.34444, -86.7179 32.4028...",01,AL,ALABAMA,01
1,Blount,01009,"POLYGON ((-86.5778 33.76532, -86.75914 33.8406...",01,AL,ALABAMA,01
2,Chambers,01017,"POLYGON ((-85.18413 32.87052, -85.12342 32.772...",01,AL,ALABAMA,01
3,Chilton,01021,"POLYGON ((-86.51734 33.02057, -86.51596 32.929...",01,AL,ALABAMA,01
4,Colbert,01033,"POLYGON ((-88.13999 34.5817, -88.13925 34.5878...",01,AL,ALABAMA,01
...,...,...,...,...,...,...,...
3216,Accomack,51001,"MULTIPOLYGON (((-75.24227 38.02721, -75.29687 ...",51,VA,VIRGINIA,51
3217,Bland,51021,"POLYGON ((-81.2251 37.23487, -81.20477 37.2430...",51,VA,VIRGINIA,51
3218,Buchanan,51027,"POLYGON ((-81.9683 37.5378, -81.92787 37.51212...",51,VA,VIRGINIA,51
3219,Charlotte,51037,"POLYGON ((-78.44332 37.0794, -78.49303 36.8912...",51,VA,VIRGINIA,51


In [14]:
merged_data = merged_data[merged_data['State_Abbreviation'] == 'TX' ]
merged_data = merged_data.drop(['State', 'State_Abbreviation', 'State_name', 'Code'], axis=1)
merged_data

,County,Fips,geometry
525,Ochiltree,48357,"POLYGON ((-100.93606 36.4996, -100.91851 36.49..."
526,Oldham,48359,"POLYGON ((-103.04238 35.18316, -103.0425 35.21..."
527,Parker,48367,"POLYGON ((-97.54418 32.99418, -97.55037 32.580..."
528,Rains,48379,"POLYGON ((-95.66539 32.96043, -95.63502 32.720..."
529,Randall,48381,"POLYGON ((-102.16884 34.74742, -102.16747 35.1..."
...,...,...,...
3089,Jefferson,48245,"POLYGON ((-94.35416 29.56146, -94.35798 29.887..."
3090,Kaufman,48257,"POLYGON ((-96.52999 32.54528, -96.52312 32.545..."
3118,Panola,48365,"POLYGON ((-94.51143 31.97398, -94.59998 31.973..."
3119,Haskell,48207,"POLYGON ((-99.47244 33.39902, -99.47126 32.957..."


In [15]:
merged_data['Fips'] = merged_data['Fips'].astype(int)

In [16]:
merged_data = pd.merge(
    merged_data,
    df,
    left_on=['County', 'Fips'],
    right_on=['COUNTY_NAME', 'COUNTYFIPS']
)
merged_data = merged_data.drop(['COUNTY_NAME', 'COUNTYFIPS'], axis=1)

merged_data

,County,Fips,geometry,Beneficiary_ID,Assessment_Effective_Date,Submitted_HIPPS_Code,Facility_Internal_ID,Age,Gender,American_Indian_or_Alaska_Native,Asian,Black_or_African_American,Hispanic_or_Latino,Native_Hawiian_or_Pacific_Islander,Unknown_Race_Ethnicity,White,ByDiscipline,DaysBetweenVisits,Days_Cared_For,READMISSION,Agency_Medicare_Number,Primary_Diagnosis_ICD_10_C_M_Code,Other_Diagnosis_Code_1_ICD_10_C_M,Other_Diagnosis_Code_2_ICD_10_C_M,Other_Diagnosis_Code_3_ICD_10_C_M,Other_Diagnosis_Code_4_ICD_10_C_M,Other_Diagnosis_Code_5_ICD_10_C_M,Weight_in_pounds,Height_in_inches,is_diabetes,is_heart_failure,is_hypertension,is_renal_failure,BMI,BMI_Category,POP_COU,HOU_COU,POP_URB,POPPCT_URB,POP_RUR,POPPCT_RUR,ACS_PCT_BACHELOR_DGR,ACS_PCT_COLLEGE_ASSOCIATE_DGR,ACS_PCT_GRADUATE_DGR,ACS_PCT_HS_GRADUATE,ACS_PCT_LT_HS,ACS_PCT_NO_WORK_NO_SCHL_16_19,ACS_PCT_POSTHS_ED,ACS_PCT_VET_BACHELOR,ACS_PCT_VET_COLLEGE,ACS_PCT_VET_HS,ACS_PCT_HH_LIMIT_ENGLISH,ACS_PCT_HH_BROADBAND,ACS_PCT_HH_BROADBAND_ANY,ACS_PCT_HH_BROADBAND_ONLY,ACS_PCT_HH_CELLULAR,ACS_PCT_HH_CELLULAR_ONLY,ACS_PCT_HH_DIAL_INTERNET_ONLY,ACS_PCT_HH_INTERNET,ACS_PCT_HH_INTERNET_NO_SUBS,ACS_PCT_HH_NO_COMP_DEV,ACS_PCT_HH_NO_INTERNET,ACS_PCT_HH_OTHER_COMP,ACS_PCT_HH_OTHER_COMP_ONLY,ACS_PCT_HH_PC,ACS_PCT_HH_PC_ONLY,ACS_PCT_HH_SAT_INTERNET,ACS_PCT_HH_SMARTPHONE,ACS_PCT_HH_SMARTPHONE_ONLY,ACS_PCT_HH_TABLET,ACS_PCT_HH_TABLET_ONLY,ACS_PCT_CHILDREN_GRANDPARENT,ACS_PCT_CHILD_1FAM,ACS_PCT_GRANDP_NO_RESPS,ACS_PCT_GRANDP_RESPS_NO_P,ACS_PCT_GRANDP_RESPS_P,ACS_PCT_HH_1PERS,ACS_PCT_HH_ABOVE65,ACS_PCT_HH_ALONE_ABOVE65,ACS_PCT_HH_KID_1PRNT,ACS_TOT_GRANDCHILDREN_GP,ACS_PCT_HEALTH_INC_138_199,ACS_PCT_HEALTH_INC_200_399,ACS_PCT_HEALTH_INC_ABOVE400,ACS_PCT_HEALTH_INC_BELOW137,ACS_PCT_HH_1FAM_FOOD_STMP,ACS_PCT_HH_FOOD_STMP,ACS_PCT_HH_FOOD_STMP_BLW_POV,ACS_PCT_HH_NO_FD_STMP_BLW_POV,ACS_PCT_HH_PUB_ASSIST,ACS_PCT_INC50,ACS_PCT_INC50_ABOVE65,ACS_PCT_NONVET_POV_18_64,ACS_PCT_PERSON_INC_100_124,ACS_PCT_PERSON_INC_125_199,ACS_PCT_PERSON_INC_ABOVE200,ACS_PCT_PERSON_INC_BELOW99,ACS_PCT_POV_AIAN,ACS_PCT_POV_ASIAN,ACS_PCT_POV_BLACK,ACS_PCT_POV_HISPANIC,ACS_PCT_POV_MULTI,ACS_PCT_POV_NHPI,ACS_PCT_POV_OTHER,ACS_PCT_POV_WHITE,ACS_PCT_VET_POV_18_64,ACS_TOT_CIVIL_NONINST_POP_POV,ACS_TOT_CIVIL_POP_POV,ACS_TOT_POP_POV,ICD_Section,ICD_Range
0,Ochiltree,48357,"POLYGON ((-100.93606 36.4996, -100.91851 36.49...",GGGGGGGS9zSDSSS,2017-10-10,3BFKS,9645,76,Male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RN,NaN,296,0,457640,M71.112,Z48.00,M54.31,I73.9,I50.9,J44.9,NaN,NaN,0,0,0,0,0,Underweight,10015.0,4332.0,8391.0,83.78,1624.0,16.22,15.910000,23.463333,3.603333,34.100000,22.926667,0.000,42.976667,17.953333,46.540000,35.506667,8.370,55.280000,82.073333,8.616667,73.020000,19.376667,0.036667,82.113333,3.976667,7.206667,13.910000,2.193333,0.000000,77.963333,5.146667,13.303333,80.883333,6.773333,61.783333,1.290000,1.880000,19.400000,6.190000,16.190000,10.953333,24.010000,24.280000,8.576667,15.466667,23.333333,14.536667,31.543333,25.836667,28.086667,48.253333,12.360000,4.780000,6.543333,12.360000,2.853333,1.640000,11.696667,10.650000,19.216667,57.376667,12.753333,0.000000,0.000,0.000000,11.973333,17.173333,0.000000,8.830000,7.813333,0.000000,3272.000000,2246.000000,3272.000000,"{'Diseases of the circulatory system', 'Factor...","{'Z00-Z99', 'I00-I99', 'M00-M99', 'J00-J99'}"
1,Ochiltree,48357,"POLYGON ((-100.93606 36.4996, -100.91851 36.49...",GGGGGGGWGeeDNDG,2017-11-15,1AGPS,9645,91,Male,0.0,0.0,0.0,0.0,0.0,NaN,1.0,RN,NaN,0,0,457640,D64.9,Z51.89,R53.1,E11.8,E16.2,R42.,114.0,68.0,1,0,0,0,18,Underweight,10015.0,4332.0,8391.0,83.78,1624.0,16.22,15.910000,23.463333,3.603333,34.100000,22.926667,0.000,42.976667,17.953333,46.540000,35.506667,8.370,55.280000,82.073333,8.616667,73.020000,19.376667,0.036667,82.113333,3.976667,7.206667,13.910000,2.193333,0.000000,77.963333,5.146667,13.303333,80.883333,6.773333,61.783333,1.290000,1.880000,19.400000,6.190000,16.190000,10.953333,24.010000,24.280000,8.576667,15.466667,23.333333,14.536667,31.543333,25.836667,28.086667,48

In [17]:
merged_data.columns.tolist()

['County',
 'Fips',
 'geometry',
 'Beneficiary_ID',
 'Assessment_Effective_Date',
 'Submitted_HIPPS_Code',
 'Facility_Internal_ID',
 'Age',
 'Gender',
 'American_Indian_or_Alaska_Native',
 'Asian',
 'Black_or_African_American',
 'Hispanic_or_Latino',
 'Native_Hawiian_or_Pacific_Islander',
 'Unknown_Race_Ethnicity',
 'White',
 'ByDiscipline',
 'DaysBetweenVisits',
 'Days_Cared_For',
 'READMISSION',
 'Agency_Medicare_Number',
 'Primary_Diagnosis_ICD_10_C_M_Code',
 'Other_Diagnosis_Code_1_ICD_10_C_M',
 'Other_Diagnosis_Code_2_ICD_10_C_M',
 'Other_Diagnosis_Code_3_ICD_10_C_M',
 'Other_Diagnosis_Code_4_ICD_10_C_M',
 'Other_Diagnosis_Code_5_ICD_10_C_M',
 'Weight_in_pounds',
 'Height_in_inches',
 'is_diabetes',
 'is_heart_failure',
 'is_hypertension',
 'is_renal_failure',
 'BMI',
 'BMI_Category',
 'POP_COU',
 'HOU_COU',
 'POP_URB',
 'POPPCT_URB',
 'POP_RUR',
 'POPPCT_RUR',
 'ACS_PCT_BACHELOR_DGR',
 'ACS_PCT_COLLEGE_ASSOCIATE_DGR',
 'ACS_PCT_GRADUATE_DGR',
 'ACS_PCT_HS_GRADUATE',
 'ACS_PCT_LT_

In [18]:
merged_data.to_csv('data/texasCountySummariesOASIS.csv', index=False)

In [28]:
merged_data_copy1 = merged_data.copy()

In [29]:
merged_data_copy2 =merged_data_copy1[['Beneficiary_ID', 'County', 'READMISSION'
 'Fips',
 'geometry',
 'Submitted_HIPPS_Code',
 'Facility_Internal_ID',
 'Age',
 'Gender',
 'American_Indian_or_Alaska_Native',
 'Asian',
 'Black_or_African_American',
 'Hispanic_or_Latino',
 'Native_Hawiian_or_Pacific_Islander',
 'Unknown_Race_Ethnicity',
 'White',
 'ByDiscipline',
 'DaysBetweenVisits',
 'Days_Cared_For',
 'Agency_Medicare_Number',
 'Primary_Diagnosis_ICD_10_C_M_Code',
 'Other_Diagnosis_Code_1_ICD_10_C_M',
 'Other_Diagnosis_Code_2_ICD_10_C_M',
 'Other_Diagnosis_Code_3_ICD_10_C_M',
 'Other_Diagnosis_Code_4_ICD_10_C_M',
 'Other_Diagnosis_Code_5_ICD_10_C_M',
 'Weight_in_pounds',
 'Height_in_inches',
 'is_diabetes',
 'is_heart_failure',
 'is_hypertension',
 'is_renal_failure',
 'BMI',
 'BMI_Category',
 'POP_COU',
 'HOU_COU',
 'POP_URB',
 'POPPCT_URB',
 'POP_RUR',
 'POPPCT_RUR',
 'ICD_Section',
 'ICD_Range']]

KeyError: "['READMISSIONFips'] not in index"

In [ ]:
merged_data_copy2

In [27]:
# Grouping the data
grouped_df = merged_data_copy2.groupby(['County', 'Fips']).agg(
    BENE_ID_unique=('Beneficiary_ID', 'nunique'),
    FACINTID_unique=('Facility_Internal_ID', 'nunique'),
    SBMHPSCD_unique=('Submitted_HIPPS_Code', 'nunique'),
    Age_mean=('Age', 'mean'),
    Gender_counts=('Gender', lambda x: x.value_counts().to_dict()),
    American_Indian_or_Alaska_Native_unique=('American_Indian_or_Alaska_Native', 'nunique'),
    Black_or_African_American_unique=('Black_or_African_American', 'nunique'),
    Hispanic_or_Latino_unique=('Hispanic_or_Latino', 'nunique'),
    Native_Hawiian_or_Pacific_Islander_unique=('Native_Hawiian_or_Pacific_Islander', 'nunique'),
    Unknown_Race_Ethnicity_unique=('American_Indian_or_Alaska_Native', 'nunique'),    
    White_unique=('White', 'nunique'),    
    ByDiscipline_counts=('ByDiscipline', lambda x: x.value_counts().to_dict()),
    M0010_counts=('Agency_Medicare_Number', lambda x: x.value_counts().to_dict()),
    DaysBetweenVisits_mean=('DaysBetweenVisits', 'mean'),
    Days_Cared_For_mean=('Days_Cared_For', 'mean'),
    BMI_Category_counts=('BMI_Category', lambda x: x.value_counts().to_dict()),
    Diabetic_patient_count=('is_diabetes', lambda x: (x == 1).sum()),
    HeartFailure_patient_count=('is_heart_failure', lambda x: (x == 1).sum()),
    Hypertension_patient_count=('is_hypertension', lambda x: (x == 1).sum()),
    RenalFailure_patient_count=('is_renal_failure', lambda x: (x == 1).sum()),
    Readmission_1_count=('READMISSION', lambda x: (x == 1).sum()),
    Readmission_0_count=('READMISSION', lambda x: (x == 0).sum()),
    geometry=('geometry', 'first'),
    POP_COU=('POP_COU', 'first'),
    HOU_COU=('HOU_COU', 'first'),
    POP_URB=('POP_URB', 'first'),
    POPPCT_URB=('POPPCT_URB', 'first'),
    POP_RUR=('POP_RUR', 'first'),
    POPPCT_RUR=('POPPCT_RUR', 'first'),
    ICD_Section=('ICD_Section', lambda x: ','.join(sorted(set(','.join(x).split(','))))),
    ICD_Range=('ICD_Range', lambda x: ','.join(sorted(set(','.join(x).split(',')))))
).reset_index()
grouped_df

KeyError: "Column(s) ['READMISSION'] do not exist"

In [21]:
grouped_df.columns

Index(['County', 'Fips', 'BENE_ID_unique', 'FACINTID_unique',
       'HHA_AGNCY_ID_unique', 'SBMHPSCD_unique', 'Age_mean', 'Gender_counts',
       'Race_counts', 'ByDiscipline_counts', 'M0010_counts',
       'DaysBetweenVisits_mean', 'Days_Cared_For_mean', 'BMI_Category_counts',
       'Diabetic_patient_count', 'HeartFailure_patient_count',
       'Hypertension_patient_count', 'Readmission_1_count',
       'Readmission_0_count', 'geometry', 'POP_COU', 'HOU_COU', 'POP_URB',
       'POPPCT_URB', 'POP_RUR', 'POPPCT_RUR', 'ICD_Section', 'ICD_Range'],
      dtype='object')

In [22]:
# Filter Texas counties
texas_counties_geojson = {
    "type": "FeatureCollection",
    "features": [feature for feature in counties['features'] if feature['properties']['STATE'] == '48']
}


In [23]:
grouped_df.to_csv('streamlit/streamlit_df.csv', index=False)

In [24]:
# Grouping the data
grouped_df = merged_data_copy1.groupby(['County', 'Fips']).agg(
    BENE_ID_unique=('BENE_ID', 'nunique'),
    FACINTID_unique=('FACINTID', 'nunique'),
    HHA_AGNCY_ID_unique=('HHA_AGNCY_ID', 'nunique'),
    SBMHPSCD_unique=('SBMHPSCD', 'nunique'),
    Diabetic_patient_count=('Diabetic_patient', lambda x: (x == 1).sum()),
    HeartFailure_patient_count=('HeartFailure_patient', lambda x: (x == 1).sum()),
    Hypertension_patient_count=('Hypertension_patient', lambda x: (x == 1).sum()),
    
    Readmission_1_count=('Readmission', lambda x: (x == 1).sum()),
    Readmission_0_count=('Readmission', lambda x: (x == 0).sum()),
    geometry=('geometry', 'first'),
    POP_COU=('POP_COU', 'first'),
    HOU_COU=('HOU_COU', 'first'),
    POP_URB=('POP_URB', 'first'),
    POPPCT_URB=('POPPCT_URB', 'first'),
    POP_RUR=('POP_RUR', 'first'),
    POPPCT_RUR=('POPPCT_RUR', 'first'),
).reset_index()
grouped_df

,County,Fips,BENE_ID_unique,FACINTID_unique,HHA_AGNCY_ID_unique,SBMHPSCD_unique,Diabetic_patient_count,HeartFailure_patient_count,Hypertension_patient_count,Readmission_1_count,Readmission_0_count,geometry,POP_COU,HOU_COU,POP_URB,POPPCT_URB,POP_RUR,POPPCT_RUR
0,Anderson,48001,338,27,27,108,407.0,151.0,330.0,109.0,298.0,"POLYGON ((-95.42851 32.08447, -95.44675 31.843...",57922.0,20131.0,18615.0,32.14,39307.0,67.86
1,Andrews,48003,53,11,11,36,66.0,17.0,53.0,20.0,46.0,"POLYGON ((-102.28705 32.08699, -102.79909 32.0...",18610.0,7020.0,15201.0,81.68,3409.0,18.32
2,Angelina,48005,569,27,27,163,657.0,179.0,536.0,152.0,505.0,"POLYGON ((-94.32662 31.22475, -94.12963 31.099...",86395.0,36480.0,41551.0,48.09,44844.0,51.91
3,Aransas,48007,131,24,24,66,143.0,30.0,111.0,34.0,109.0,"MULTIPOLYGON (((-96.85345 28.06134, -96.92905 ...",23830.0,15500.0,18212.0,76.42,5618.0,23.58
4,Archer,48009,62,15,15,42,69.0,25.0,51.0,16.0,53.0,"POLYGON ((-98.95309 33.83400, -98.74236 33.834...",8560.0,3865.0,1082.0,12.64,7478.0,87.36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243,Wood,48499,361,42,42,119,418.0,133.0,341.0,93.0,325.0,"POLYGON ((-95.15341 32.57012, -95.16752 32.543...",44843.0,22129.0,5699.0,12.71,39144.0,87.29
244,Yoakum,48501,26,4,4,17,32.0,12.0,28.0,11.0,21.0,"POLYGON ((-102.59502 32.95883, -103.06466 32.9...",7694.0,3009.0,0.0,0.00,7694.0,100.00
245,Young,48503,161,11,11,78,190.0,70.0,113.0,50.0,140.0,"POLYGON ((-98.95087 32.95692, -98.95394 33.397...",17867.0,8539.0,8585.0,48.05,9282.0,51.95
246,Zapata,48505,143,17,17,48,163.0,14.0,138.0,42.0,121.0,"POLYGON ((-98.95423 26.78569, -99.01111 26.675...",13889.0,6159.0,10942.0,78.78,2947.0,21.22


In [25]:
sdoh =merged_data_copy1[['County',
 'Fips',
 'ACS_PCT_BACHELOR_DGR',
 'ACS_PCT_COLLEGE_ASSOCIATE_DGR',
 'ACS_PCT_GRADUATE_DGR',
 'ACS_PCT_HS_GRADUATE',
 'ACS_PCT_LT_HS',
 'ACS_PCT_NO_WORK_NO_SCHL_16_19',
 'ACS_PCT_POSTHS_ED',
 'ACS_PCT_VET_BACHELOR',
 'ACS_PCT_VET_COLLEGE',
 'ACS_PCT_VET_HS',
 'ACS_PCT_HH_LIMIT_ENGLISH',
 'ACS_PCT_HH_BROADBAND',
 'ACS_PCT_HH_BROADBAND_ANY',
 'ACS_PCT_HH_BROADBAND_ONLY',
 'ACS_PCT_HH_CELLULAR',
 'ACS_PCT_HH_CELLULAR_ONLY',
 'ACS_PCT_HH_DIAL_INTERNET_ONLY',
 'ACS_PCT_HH_INTERNET',
 'ACS_PCT_HH_INTERNET_NO_SUBS',
 'ACS_PCT_HH_NO_COMP_DEV',
 'ACS_PCT_HH_NO_INTERNET',
 'ACS_PCT_HH_OTHER_COMP',
 'ACS_PCT_HH_OTHER_COMP_ONLY',
 'ACS_PCT_HH_PC',
 'ACS_PCT_HH_PC_ONLY',
 'ACS_PCT_HH_SAT_INTERNET',
 'ACS_PCT_HH_SMARTPHONE',
 'ACS_PCT_HH_SMARTPHONE_ONLY',
 'ACS_PCT_HH_TABLET',
 'ACS_PCT_HH_TABLET_ONLY',
 'ACS_PCT_CHILDREN_GRANDPARENT',
 'ACS_PCT_CHILD_1FAM',
 'ACS_PCT_GRANDP_NO_RESPS',
 'ACS_PCT_GRANDP_RESPS_NO_P',
 'ACS_PCT_GRANDP_RESPS_P',
 'ACS_PCT_HH_1PERS',
 'ACS_PCT_HH_ABOVE65',
 'ACS_PCT_HH_ALONE_ABOVE65',
 'ACS_PCT_HH_KID_1PRNT',
 'ACS_TOT_GRANDCHILDREN_GP',
 'ACS_PCT_HEALTH_INC_138_199',
 'ACS_PCT_HEALTH_INC_200_399',
 'ACS_PCT_HEALTH_INC_ABOVE400',
 'ACS_PCT_HEALTH_INC_BELOW137',
 'ACS_PCT_HH_1FAM_FOOD_STMP',
 'ACS_PCT_HH_FOOD_STMP',
 'ACS_PCT_HH_FOOD_STMP_BLW_POV',
 'ACS_PCT_HH_NO_FD_STMP_BLW_POV',
 'ACS_PCT_HH_PUB_ASSIST',
 'ACS_PCT_INC50',
 'ACS_PCT_INC50_ABOVE65',
 'ACS_PCT_NONVET_POV_18_64',
 'ACS_PCT_PERSON_INC_100_124',
 'ACS_PCT_PERSON_INC_125_199',
 'ACS_PCT_PERSON_INC_ABOVE200',
 'ACS_PCT_PERSON_INC_BELOW99',
 'ACS_PCT_POV_AIAN',
 'ACS_PCT_POV_ASIAN',
 'ACS_PCT_POV_BLACK',
 'ACS_PCT_POV_HISPANIC',
 'ACS_PCT_POV_MULTI',
 'ACS_PCT_POV_NHPI',
 'ACS_PCT_POV_OTHER',
 'ACS_PCT_POV_WHITE',
 'ACS_PCT_VET_POV_18_64',
 'ACS_TOT_CIVIL_NONINST_POP_POV',
 'ACS_TOT_CIVIL_POP_POV',
 'ACS_TOT_POP_POV',
]]

In [26]:
df = grouped_df.merge(sdoh, on=['County', 'Fips']).drop_duplicates()
df

,County,Fips,BENE_ID_unique,FACINTID_unique,HHA_AGNCY_ID_unique,SBMHPSCD_unique,Diabetic_patient_count,HeartFailure_patient_count,Hypertension_patient_count,Readmission_1_count,Readmission_0_count,geometry,POP_COU,HOU_COU,POP_URB,POPPCT_URB,POP_RUR,POPPCT_RUR,ACS_PCT_BACHELOR_DGR,ACS_PCT_COLLEGE_ASSOCIATE_DGR,ACS_PCT_GRADUATE_DGR,ACS_PCT_HS_GRADUATE,ACS_PCT_LT_HS,ACS_PCT_NO_WORK_NO_SCHL_16_19,ACS_PCT_POSTHS_ED,ACS_PCT_VET_BACHELOR,ACS_PCT_VET_COLLEGE,ACS_PCT_VET_HS,ACS_PCT_HH_LIMIT_ENGLISH,ACS_PCT_HH_BROADBAND,ACS_PCT_HH_BROADBAND_ANY,ACS_PCT_HH_BROADBAND_ONLY,ACS_PCT_HH_CELLULAR,ACS_PCT_HH_CELLULAR_ONLY,ACS_PCT_HH_DIAL_INTERNET_ONLY,ACS_PCT_HH_INTERNET,ACS_PCT_HH_INTERNET_NO_SUBS,ACS_PCT_HH_NO_COMP_DEV,ACS_PCT_HH_NO_INTERNET,ACS_PCT_HH_OTHER_COMP,ACS_PCT_HH_OTHER_COMP_ONLY,ACS_PCT_HH_PC,ACS_PCT_HH_PC_ONLY,ACS_PCT_HH_SAT_INTERNET,ACS_PCT_HH_SMARTPHONE,ACS_PCT_HH_SMARTPHONE_ONLY,ACS_PCT_HH_TABLET,ACS_PCT_HH_TABLET_ONLY,ACS_PCT_CHILDREN_GRANDPARENT,ACS_PCT_CHILD_1FAM,ACS_PCT_GRANDP_NO_RESPS,ACS_PCT_GRANDP_RESPS_NO_P,ACS_PCT_GRANDP_RESPS_P,ACS_PCT_HH_1PERS,ACS_PCT_HH_ABOVE65,ACS_PCT_HH_ALONE_ABOVE65,ACS_PCT_HH_KID_1PRNT,ACS_TOT_GRANDCHILDREN_GP,ACS_PCT_HEALTH_INC_138_199,ACS_PCT_HEALTH_INC_200_399,ACS_PCT_HEALTH_INC_ABOVE400,ACS_PCT_HEALTH_INC_BELOW137,ACS_PCT_HH_1FAM_FOOD_STMP,ACS_PCT_HH_FOOD_STMP,ACS_PCT_HH_FOOD_STMP_BLW_POV,ACS_PCT_HH_NO_FD_STMP_BLW_POV,ACS_PCT_HH_PUB_ASSIST,ACS_PCT_INC50,ACS_PCT_INC50_ABOVE65,ACS_PCT_NONVET_POV_18_64,ACS_PCT_PERSON_INC_100_124,ACS_PCT_PERSON_INC_125_199,ACS_PCT_PERSON_INC_ABOVE200,ACS_PCT_PERSON_INC_BELOW99,ACS_PCT_POV_AIAN,ACS_PCT_POV_ASIAN,ACS_PCT_POV_BLACK,ACS_PCT_POV_HISPANIC,ACS_PCT_POV_MULTI,ACS_PCT_POV_NHPI,ACS_PCT_POV_OTHER,ACS_PCT_POV_WHITE,ACS_PCT_VET_POV_18_64,ACS_TOT_CIVIL_NONINST_POP_POV,ACS_TOT_CIVIL_POP_POV,ACS_TOT_POP_POV
0,Anderson,48001,338,27,27,108,407.0,151.0,330.0,109.0,298.0,"POLYGON ((-95.42851 32.08447, -95.44675 31.843...",57922.0,20131.0,18615.0,32.14,39307.0,67.86,7.841667,32.505833,4.401667,38.857500,16.396667,0.484167,44.747500,21.754168,43.486668,34.760000,1.988333,45.035830,67.546670,12.244166,52.398335,13.515000,0.416667,67.963330,2.157500,11.767500,21.545834,6.010000,0.0000,61.189167,4.223333,16.269167,72.36000,12.733334,46.878334,0.625000,10.084167,30.835000,37.776670,20.465834,33.424168,23.509167,29.446667,10.115833,16.695833,103.250000,19.629168,32.141666,21.355000,26.874166,27.555000,10.730833,4.674167,6.685834,12.410833,4.845833,2.404167,18.850000,4.595833,21.844166,53.496666,20.064167,18.616667,9.615000,22.032500,21.848333,22.510834,0.0,10.224167,9.229167,2.745000,3700.1667,2778.0833,3700.1667
407,Andrews,48003,53,11,11,36,66.0,17.0,53.0,20.0,46.0,"POLYGON ((-102.28705 32.08699, -102.79909 32.0...",18610.0,7020.0,15201.0,81.68,3409.0,18.32,11.447500,27.912500,4.192500,33.095000,23.350000,0.000000,43.550000,28.932500,31.685000,39.382500,8.120000,63.585000,89.395000,6.152500,80.537500,18.565000,0.127500,89.522500,0.722500,6.565000,9.755000,1.482500,0.2375,70.922500,3.365000,8.900000,85.82500,12.707500,63.812500,0.185000,8.790000,27.960000,47.802500,19.100000,33.097500,17.877500,22.322500,7.277500,16.767500,102.500000,9.615000,35.280000,36.475000,18.632500,11.412500,7.582500,3.515000,6.657500,7.582500,7.362500,3.050000,9.477500,5.740000,12.320000,71.752500,10.190000,0.000000,0.000000,0.000000,16.182500,2.860000,0.0,6.445000,3.332500,4.860000,4529.5000,3137.0000,4529.5000
473,Angelina,48005,569,27,27,163,657.0,179.0,536.0,152.0,505.0,"POLYGON ((-94.32662 31.22475, -94.12963 31.099...",86395.0,36480.0,41551.0,48.09,44844.0,51.91,12.648571,33.956192,5.838095,30.977620,16.579048,4.775238,52.443810,18.338572,41.577618,40.083332,2.256667,63.463333,84.765720,8.862381,74.194280,14.499524,0.292857,85.059525,1.530000,10.047143,13.410953,2.098571,0.0000,70.207146,5.489524,6.886667,80.57333,10.967143,59.636192,1.046191,12.400476,35.436670,45.828094,21.942858,32.228573,23.362858,31.778095,9.943334,23.423810,120.857140,15.026667,31.463810,25.890953,27.619524,42.563330,1

In [27]:
print(df['geometry'].dtype)

geometry


In [28]:
topics = {
    'Attainment':['ACS_PCT_BACHELOR_DGR', 'ACS_PCT_COLLEGE_ASSOCIATE_DGR', 'ACS_PCT_GRADUATE_DGR', 'ACS_PCT_HS_GRADUATE', 'ACS_PCT_LT_HS',
                  'ACS_PCT_NO_WORK_NO_SCHL_16_19', 'ACS_PCT_POSTHS_ED', 'ACS_PCT_VET_BACHELOR', 'ACS_PCT_VET_COLLEGE', 'ACS_PCT_VET_HS'],
    'English Fluency':[ 'ACS_PCT_HH_LIMIT_ENGLISH' ],
    'Internet Connectivity'	:['ACS_PCT_HH_BROADBAND', 'ACS_PCT_HH_BROADBAND_ANY', 'ACS_PCT_HH_BROADBAND_ONLY', 'ACS_PCT_HH_CELLULAR', 'ACS_PCT_HH_CELLULAR_ONLY',
                              'ACS_PCT_HH_DIAL_INTERNET_ONLY', 'ACS_PCT_HH_INTERNET', 'ACS_PCT_HH_INTERNET_NO_SUBS', 'ACS_PCT_HH_NO_COMP_DEV', 'ACS_PCT_HH_NO_INTERNET',
                              'ACS_PCT_HH_OTHER_COMP', 'ACS_PCT_HH_OTHER_COMP_ONLY', 'ACS_PCT_HH_PC', 'ACS_PCT_HH_PC_ONLY', 'ACS_PCT_HH_SAT_INTERNET', 'ACS_PCT_HH_SMARTPHONE', 'ACS_PCT_HH_SMARTPHONE_ONLY', 'ACS_PCT_HH_TABLET', 'ACS_PCT_HH_TABLET_ONLY'],
    'Living Conditions'	:['ACS_PCT_CHILDREN_GRANDPARENT', 'ACS_PCT_CHILD_1FAM', 'ACS_PCT_GRANDP_NO_RESPS', 'ACS_PCT_GRANDP_RESPS_NO_P', 'ACS_PCT_GRANDP_RESPS_P', 
                          'ACS_PCT_HH_1PERS', 'ACS_PCT_HH_ABOVE65', 'ACS_PCT_HH_ALONE_ABOVE65', 'ACS_PCT_HH_KID_1PRNT', 'ACS_TOT_GRANDCHILDREN_GP'],
    'Poverty':['ACS_PCT_HEALTH_INC_138_199', 'ACS_PCT_HEALTH_INC_200_399', 'ACS_PCT_HEALTH_INC_ABOVE400', 'ACS_PCT_HEALTH_INC_BELOW137', 'ACS_PCT_HH_1FAM_FOOD_STMP',
                  'ACS_PCT_HH_FOOD_STMP', 'ACS_PCT_HH_FOOD_STMP_BLW_POV', 'ACS_PCT_HH_NO_FD_STMP_BLW_POV', 'ACS_PCT_HH_PUB_ASSIST', 'ACS_PCT_INC50', 'ACS_PCT_INC50_ABOVE65', 
                  'ACS_PCT_NONVET_POV_18_64', 'ACS_PCT_PERSON_INC_100_124', 'ACS_PCT_PERSON_INC_125_199', 'ACS_PCT_PERSON_INC_ABOVE200', 'ACS_PCT_PERSON_INC_BELOW99', 
                  'ACS_PCT_POV_AIAN', 'ACS_PCT_POV_ASIAN', 'ACS_PCT_POV_BLACK', 'ACS_PCT_POV_HISPANIC', 'ACS_PCT_POV_MULTI', 'ACS_PCT_POV_NHPI', 'ACS_PCT_POV_OTHER', 
                  'ACS_PCT_POV_WHITE', 'ACS_PCT_VET_POV_18_64', 'ACS_TOT_CIVIL_NONINST_POP_POV', 'ACS_TOT_CIVIL_POP_POV', 'ACS_TOT_POP_POV'],
}

In [35]:
disparity_results = []

# Iterate through the topics and calculate disparity metrics for each county
for topic, indicators in topics.items():
    for indicator in indicators:
        # Calculate the overall average of the indicator
        avg_value = df[indicator].mean()

        for index, row in df.iterrows():
            # Calculate Absolute Index of Disparity (AID) for the county
            aid = abs(row[indicator] - avg_value)
            
            # Calculate Relative Index of Disparity (RID) for the county
            rid = aid / avg_value
            
            # Calculate Population-Weighted Index of Disparity (PWID) for the county
            pwid = abs(row[indicator] - avg_value) * row['POP_COU'] / df['POP_COU'].sum()
            
            # Calculate Population Attributable Proportion (PAP) for the county (urban vs rural)
            urban_disparity = abs(row[indicator] - avg_value) * row['POP_COU'] if row['POPPCT_URB'] > 0.5 else 0
            rural_disparity = abs(row[indicator] - avg_value) * row['POP_COU'] if row['POPPCT_RUR'] > 0.5 else 0
            total_disparity = abs(row[indicator] - avg_value) * row['POP_COU']
            pap = urban_disparity / total_disparity if total_disparity > 0 else 0
            
            # Store the results for each county
            disparity_results.append({
                'County': row['County'],
                'Topic': topic,
                'Indicator': indicator,
                'AbsoluteIndexDisparity': aid,
                'RelativeIndexDisparity': rid,
                'PopulationWeightedIndexDisparity': pwid,
                'PopulationAttributableProportion': pap
            })
disparity_df = pd.DataFrame(disparity_results)
disparity_df

,County,Topic,Indicator,AbsoluteIndexDisparity,RelativeIndexDisparity,PopulationWeightedIndexDisparity,PopulationAttributableProportion
0,Anderson,Attainment,ACS_PCT_BACHELOR_DGR,5.715634,0.421591,0.011360,1.0
1,Andrews,Attainment,ACS_PCT_BACHELOR_DGR,2.109801,0.155621,0.001347,1.0
2,Angelina,Attainment,ACS_PCT_BACHELOR_DGR,0.908730,0.067029,0.002694,1.0
3,Aransas,Attainment,ACS_PCT_BACHELOR_DGR,1.690880,0.124721,0.001383,1.0
4,Archer,Attainment,ACS_PCT_BACHELOR_DGR,0.606032,0.044702,0.000178,1.0
...,...,...,...,...,...,...,...
16859,Wood,Poverty,ACS_TOT_POP_POV,358.141819,0.110002,0.551103,1.0
16860,Yoakum,Poverty,ACS_TOT_POP_POV,1050.225119,0.322573,0.277279,0.0
16861,Young,Poverty,ACS_TOT_POP_POV,274.425119,0.084289,0.168251,1.0
16862,Zapata,Poverty,ACS_TOT_POP_POV,914.108181,0.280765,0.435664,1.0


In [36]:
disparity_df['Indicator'] = disparity_df['Indicator'].str.replace('ACS_PCT', '%')
disparity_df['Indicator'] = disparity_df['Indicator'].str.replace('ACS_TOT', 'TOTAL').str.replace('_', ' ')

disparity_df

,County,Topic,Indicator,AbsoluteIndexDisparity,RelativeIndexDisparity,PopulationWeightedIndexDisparity,PopulationAttributableProportion
0,Anderson,Attainment,% BACHELOR DGR,5.715634,0.421591,0.011360,1.0
1,Andrews,Attainment,% BACHELOR DGR,2.109801,0.155621,0.001347,1.0
2,Angelina,Attainment,% BACHELOR DGR,0.908730,0.067029,0.002694,1.0
3,Aransas,Attainment,% BACHELOR DGR,1.690880,0.124721,0.001383,1.0
4,Archer,Attainment,% BACHELOR DGR,0.606032,0.044702,0.000178,1.0
...,...,...,...,...,...,...,...
16859,Wood,Poverty,TOTAL POP POV,358.141819,0.110002,0.551103,1.0
16860,Yoakum,Poverty,TOTAL POP POV,1050.225119,0.322573,0.277279,0.0
16861,Young,Poverty,TOTAL POP POV,274.425119,0.084289,0.168251,1.0
16862,Zapata,Poverty,TOTAL POP POV,914.108181,0.280765,0.435664,1.0


In [38]:
disparity_df['Indicator'].unique()

array(['% BACHELOR DGR', '% COLLEGE ASSOCIATE DGR', '% GRADUATE DGR',
       '% HS GRADUATE', '% LT HS', '% NO WORK NO SCHL 16 19',
       '% POSTHS ED', '% VET BACHELOR', '% VET COLLEGE', '% VET HS',
       '% HH LIMIT ENGLISH', '% HH BROADBAND', '% HH BROADBAND ANY',
       '% HH BROADBAND ONLY', '% HH CELLULAR', '% HH CELLULAR ONLY',
       '% HH DIAL INTERNET ONLY', '% HH INTERNET',
       '% HH INTERNET NO SUBS', '% HH NO COMP DEV', '% HH NO INTERNET',
       '% HH OTHER COMP', '% HH OTHER COMP ONLY', '% HH PC',
       '% HH PC ONLY', '% HH SAT INTERNET', '% HH SMARTPHONE',
       '% HH SMARTPHONE ONLY', '% HH TABLET', '% HH TABLET ONLY',
       '% CHILDREN GRANDPARENT', '% CHILD 1FAM', '% GRANDP NO RESPS',
       '% GRANDP RESPS NO P', '% GRANDP RESPS P', '% HH 1PERS',
       '% HH ABOVE65', '% HH ALONE ABOVE65', '% HH KID 1PRNT',
       'TOTAL GRANDCHILDREN GP', '% HEALTH INC 138 199',
       '% HEALTH INC 200 399', '% HEALTH INC ABOVE400',
       '% HEALTH INC BELOW137', '% HH 

In [37]:
disparity_df.to_csv('streamlit/streamlit_sdoh.csv', index=False)